In [ ]:
# ============================================================
# mBERT Encoder-Decoder — Nepali Abstractive Summarization
# ============================================================
# Dependencies (install once):
#   pip install torch pandas numpy scikit-learn transformers
#               datasets sacrebleu bert-score tqdm
#
# Environment:
#   NEPALI_DATA_DIR   folder containing small.csv, medium.csv, large.csv
#                     (default: ./data/)
#   NEPALI_WORK_DIR   output folder (default: ./mbert_workspace/)
#
# Generation is run TWICE:
#   v1 — original decoding (beam=4)                  → baseline
#   v2 — tuned decoding (beam=5, length_penalty=2.0,
#        min_length=40, no_repeat_ngram=3)            → reported config
# Both prediction files and both metric tables are saved.
#
# Special tokens (required for BERT-based EncoderDecoder):
#   decoder_start_token_id = [CLS]
#   eos_token_id           = [SEP]
#   pad_token_id           = [PAD]
# ============================================================

import os
import time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from datasets import Dataset as HFDataset
from transformers import (
    AutoTokenizer,
    EncoderDecoderModel,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from sacrebleu.metrics import BLEU
from bert_score import score as bert_score_fn
from tqdm.auto import tqdm


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
CONFIG = {
    "data_dir":   Path(os.environ.get("NEPALI_DATA_DIR", "./data")),
    "work_dir":   Path(os.environ.get("NEPALI_WORK_DIR", "./mbert_workspace")),
    "base_model": "bert-base-multilingual-cased",

    "test_size":      0.2,
    "seed":           42,
    "max_input_len":  512,
    "max_target_len": 128,

    # Training — unchanged from your script
    "train_batch_size":     8,
    "eval_batch_size":      8,
    "grad_accum":           2,
    "epochs":               3,
    "lr":                   3e-5,
    "warmup_steps":         500,
    "weight_decay":         0.01,
    "fp16":                 True,
    "logging_steps":        500,
    "generation_max_len":   128,

    # v1 — original decoding (unchanged)
    "v1_max_len":    128,
    "v1_num_beams":  4,

    # v2 — tuned decoding (unchanged)
    "v2_max_len":        128,
    "v2_min_len":        40,
    "v2_num_beams":      5,
    "v2_length_penalty": 2.0,
    "v2_no_repeat":      3,

    # Inference
    "infer_batch_size":  8,
    "infer_num_workers": 2,
}


# ------------------------------------------------------------
# Reproducibility and runtime
# ------------------------------------------------------------
import random
random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["seed"])
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("CUDA GPU required.")

WORK_DIR        = CONFIG["work_dir"]
WORK_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR        = WORK_DIR / "mbert-checkpoints"
FINAL_MODEL_DIR = WORK_DIR / "mbert-final"
TEST_RAW_CSV    = WORK_DIR / "mbert_test_raw.csv"
PRED_V1_CSV     = WORK_DIR / "mbert_test_predictions_v1.csv"
PRED_V2_CSV     = WORK_DIR / "mbert_test_predictions_v2.csv"
SCORES_CSV      = WORK_DIR / "mbert_final_scores.csv"


# ------------------------------------------------------------
# Data: load, clean, split
# ------------------------------------------------------------
def load_and_split(cfg):
    data_dir = cfg["data_dir"]
    required = ("small.csv", "medium.csv", "large.csv")
    missing = [n for n in required if not (data_dir / n).is_file()]
    if missing:
        raise FileNotFoundError(
            f"Missing {missing} in {data_dir.resolve()}. "
            f"Set NEPALI_DATA_DIR or place the CSVs in ./data/."
        )

    df = pd.concat([pd.read_csv(data_dir / n) for n in required], ignore_index=True)
    df = df.dropna(subset=["content", "summary"]).copy()
    df["content"] = df["content"].astype(str).str.strip()
    df["summary"] = df["summary"].astype(str).str.strip()
    df = df[(df["content"] != "") & (df["summary"] != "")]
    df = df.drop_duplicates(subset=["content"], keep="first").reset_index(drop=True)
    print(f"Cleaned rows: {len(df)}")

    train_df, temp_df = train_test_split(
        df, test_size=cfg["test_size"], random_state=cfg["seed"]
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.5, random_state=cfg["seed"]
    )
    print(f"Train/Val/Test: {len(train_df)}/{len(val_df)}/{len(test_df)}")
    return train_df, val_df, test_df


train_df, val_df, test_df = load_and_split(CONFIG)
test_df[["content", "summary"]].to_csv(TEST_RAW_CSV, index=False, encoding="utf-8")


# ------------------------------------------------------------
# Tokenizer
# ------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(CONFIG["base_model"])
print(f"Tokenizer: {type(tokenizer).__name__}, vocab={tokenizer.vocab_size}")
print(f"cls={tokenizer.cls_token_id} sep={tokenizer.sep_token_id} pad={tokenizer.pad_token_id}")


# ------------------------------------------------------------
# Load or train model
# ------------------------------------------------------------
if (FINAL_MODEL_DIR / "config.json").is_file():
    print(f"Loading trained model from {FINAL_MODEL_DIR}")
    model = EncoderDecoderModel.from_pretrained(FINAL_MODEL_DIR).to(device)
    tokenizer = AutoTokenizer.from_pretrained(FINAL_MODEL_DIR)
else:
    print("Training from scratch...")

    def tokenize_fn(examples):
        inputs = tokenizer(
            examples["content"],
            max_length=CONFIG["max_input_len"],
            truncation=True,
            padding=False,
        )
        labels = tokenizer(
            examples["summary"],
            max_length=CONFIG["max_target_len"],
            truncation=True,
            padding=False,
        )
        inputs["labels"] = labels["input_ids"]
        return inputs

    train_ds = HFDataset.from_pandas(train_df[["content", "summary"]])
    val_ds   = HFDataset.from_pandas(val_df[["content", "summary"]])
    tokenized_train = train_ds.map(tokenize_fn, batched=True, remove_columns=["content", "summary"])
    tokenized_val   = val_ds.map(tokenize_fn, batched=True, remove_columns=["content", "summary"])

    model = EncoderDecoderModel.from_encoder_decoder_pretrained(
        CONFIG["base_model"], CONFIG["base_model"]
    )
    model.config.decoder_start_token_id = tokenizer.cls_token_id
    model.config.eos_token_id           = tokenizer.sep_token_id
    model.config.pad_token_id           = tokenizer.pad_token_id
    print(f"Parameters: {model.num_parameters() / 1e6:.1f}M")

    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer, model=model, padding=True,
    )

    training_args = Seq2SeqTrainingArguments(
        output_dir=str(CKPT_DIR),
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=CONFIG["logging_steps"],
        per_device_train_batch_size=CONFIG["train_batch_size"],
        per_device_eval_batch_size=CONFIG["eval_batch_size"],
        gradient_accumulation_steps=CONFIG["grad_accum"],
        num_train_epochs=CONFIG["epochs"],
        learning_rate=CONFIG["lr"],
        warmup_steps=CONFIG["warmup_steps"],
        weight_decay=CONFIG["weight_decay"],
        fp16=CONFIG["fp16"],
        predict_with_generate=True,
        generation_max_length=CONFIG["generation_max_len"],
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to="none",
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        data_collator=data_collator,
    )
    try:
        trainer.tokenizer = tokenizer
    except AttributeError:
        pass

    trainer.train()
    model.save_pretrained(FINAL_MODEL_DIR)
    tokenizer.save_pretrained(FINAL_MODEL_DIR)
    print(f"Saved -> {FINAL_MODEL_DIR}")

model = model.to(device).eval()
model.config.decoder_start_token_id = tokenizer.cls_token_id
model.config.eos_token_id           = tokenizer.sep_token_id
model.config.pad_token_id           = tokenizer.pad_token_id


# ------------------------------------------------------------
# Batch generation
# ------------------------------------------------------------
class ContentDataset(Dataset):
    def __init__(self, frame):
        self.texts = frame["content"].astype(str).tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        return self.texts[i]


def make_collate(tok, cfg):
    def collate(batch):
        return tok(
            batch,
            max_length=cfg["max_input_len"],
            truncation=True,
            padding=True,
            return_tensors="pt",
        )
    return collate


def generate_predictions(model, tokenizer, test_frame, gen_kwargs, cfg, desc):
    """Run batch generation over the test set and return decoded strings."""
    loader = DataLoader(
        ContentDataset(test_frame),
        batch_size=cfg["infer_batch_size"],
        shuffle=False,
        collate_fn=make_collate(tokenizer, cfg),
        num_workers=cfg["infer_num_workers"],
        pin_memory=True,
    )
    preds = []
    for batch in tqdm(loader, desc=desc):
        with torch.no_grad():
            gen_ids = model.generate(
                batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.sep_token_id,
                decoder_start_token_id=tokenizer.cls_token_id,
                **gen_kwargs,
            )
        preds.extend(tokenizer.batch_decode(gen_ids, skip_special_tokens=True))
    return preds


# v1 — original decoding
v1_kwargs = dict(
    max_length=CONFIG["v1_max_len"],
    num_beams=CONFIG["v1_num_beams"],
    early_stopping=True,
)
t0 = time.time()
v1_preds = generate_predictions(model, tokenizer, test_df, v1_kwargs, CONFIG, "Generating v1")
print(f"v1: {len(v1_preds)} predictions in {(time.time() - t0) / 60:.1f} min")

pd.DataFrame({
    "source":    test_df["content"].astype(str),
    "reference": test_df["summary"].astype(str),
    "predicted": v1_preds,
}).to_csv(PRED_V1_CSV, index=False, encoding="utf-8")
print(f"Saved v1 -> {PRED_V1_CSV}")

# v2 — tuned decoding
v2_kwargs = dict(
    max_length=CONFIG["v2_max_len"],
    min_length=CONFIG["v2_min_len"],
    num_beams=CONFIG["v2_num_beams"],
    length_penalty=CONFIG["v2_length_penalty"],
    no_repeat_ngram_size=CONFIG["v2_no_repeat"],
    early_stopping=True,
)
t0 = time.time()
v2_preds = generate_predictions(model, tokenizer, test_df, v2_kwargs, CONFIG, "Generating v2")
print(f"v2: {len(v2_preds)} predictions in {(time.time() - t0) / 60:.1f} min")

pd.DataFrame({
    "source":    test_df["content"].astype(str),
    "reference": test_df["summary"].astype(str),
    "predicted": v2_preds,
}).to_csv(PRED_V2_CSV, index=False, encoding="utf-8")
print(f"Saved v2 -> {PRED_V2_CSV}")


# ------------------------------------------------------------
# Scoring helpers
# ------------------------------------------------------------
def _ngrams(tokens, n):
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]


def rouge_n_f1(hyp, ref, n):
    h_ng, r_ng = _ngrams(hyp, n), _ngrams(ref, n)
    if not h_ng or not r_ng:
        return 0.0
    overlap = sum((Counter(h_ng) & Counter(r_ng)).values())
    p, r = overlap / len(h_ng), overlap / len(r_ng)
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)


def rouge_l_f1(hyp, ref):
    m, n = len(hyp), len(ref)
    if m == 0 or n == 0:
        return 0.0
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            dp[i][j] = (
                dp[i - 1][j - 1] + 1 if hyp[i - 1] == ref[j - 1]
                else max(dp[i - 1][j], dp[i][j - 1])
            )
    lcs = dp[m][n]
    p, r = lcs / m, lcs / n
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)


def jaccard_word(a, b):
    A, B = set(str(a).split()), set(str(b).split())
    if not A and not B:
        return 1.0
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)


def jaccard_char_bigram(a, b):
    a, b = str(a), str(b)
    A = {a[i:i + 2] for i in range(max(len(a) - 1, 0))}
    B = {b[i:i + 2] for i in range(max(len(b) - 1, 0))}
    if not A and not B:
        return 1.0
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)


def score(pred_csv):
    """Compute the full metric suite for one predictions CSV."""
    df = pd.read_csv(pred_csv).fillna("")
    empty = df["predicted"].astype(str).str.strip() == ""
    n_empty = int(empty.sum())
    if n_empty:
        df.loc[empty, "predicted"] = "।"

    refs = df["reference"].astype(str).tolist()
    hyps = df["predicted"].astype(str).tolist()

    bleu_f  = BLEU(tokenize="flores200", max_ngram_order=4).corpus_score(hyps, [refs]).score
    bleu_13 = BLEU(tokenize="13a",      max_ngram_order=4).corpus_score(hyps, [refs]).score

    pairs = [(h.split(), r.split()) for h, r in zip(hyps, refs)]
    r1 = float(np.mean([rouge_n_f1(h, r, 1) for h, r in pairs]))
    r2 = float(np.mean([rouge_n_f1(h, r, 2) for h, r in pairs]))
    rL = float(np.mean([rouge_l_f1(h, r)    for h, r in pairs]))

    P, R, F1 = bert_score_fn(hyps, refs, lang="ne", verbose=False, device=device.type)

    jw = float(np.mean([jaccard_word(h, r)       for h, r in zip(hyps, refs)]))
    jc = float(np.mean([jaccard_char_bigram(h, r) for h, r in zip(hyps, refs)]))

    return {
        "sacrebleu_flores200": round(bleu_f,  4),
        "sacrebleu_13a":       round(bleu_13, 4),
        "rouge1":              round(r1, 4),
        "rouge2":              round(r2, 4),
        "rougeL":              round(rL, 4),
        "bertscore_p":         round(P.mean().item(),  4),
        "bertscore_r":         round(R.mean().item(),  4),
        "bertscore_f1":        round(F1.mean().item(), 4),
        "jaccard_word":        round(jw, 4),
        "jaccard_char":        round(jc, 4),
        "avg_pred_chars":      round(float(np.mean([len(h) for h in hyps])), 1),
        "n_samples":           len(refs),
        "n_empty":             n_empty,
    }


s1 = score(PRED_V1_CSV)
s2 = score(PRED_V2_CSV)


# ------------------------------------------------------------
# Print side-by-side comparison
# ------------------------------------------------------------
print("\n" + "=" * 62)
print("  mBERT Encoder-Decoder — FINAL METRICS")
print("=" * 62)
print(f"  {'Metric':<24} {'v1 (beam=4)':>16} {'v2 (tuned)':>16}")
print("-" * 62)
for key in s1:
    if key in ("n_samples", "n_empty"):
        continue
    print(f"  {key:<24} {s1[key]:>16} {s2[key]:>16}")
print("-" * 62)
print(f"  {'n_samples':<24} {s1['n_samples']:>16} {s2['n_samples']:>16}")
print(f"  {'n_empty':<24} {s1['n_empty']:>16} {s2['n_empty']:>16}")
print("=" * 62)


# ------------------------------------------------------------
# Supplementary: BLEU tokenizer ablation for v2 (for paper supplement)
# ------------------------------------------------------------
print("\n" + "=" * 62)
print("  BLEU tokenizer ablation — v2 predictions")
print("=" * 62)
_df = pd.read_csv(PRED_V2_CSV).fillna("")
_df.loc[_df["predicted"].astype(str).str.strip() == "", "predicted"] = "।"
_v2_refs = _df["reference"].astype(str).tolist()
_v2_hyps = _df["predicted"].astype(str).tolist()

for tok_name in ("13a", "intl", "flores200", "char"):
    try:
        s = BLEU(tokenize=tok_name, max_ngram_order=4).corpus_score(_v2_hyps, [_v2_refs]).score
        print(f"  BLEU-4  tokenize={tok_name:<10}  {s:>8.4f}")
    except Exception as exc:
        print(f"  BLEU-4  tokenize={tok_name:<10}  ERROR: {exc}")
print("=" * 62)


# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------
pd.DataFrame([
    {"model": "mBERT-EncDec (v1, beam=4)",   **s1},
    {"model": "mBERT-EncDec (v2, tuned)",    **s2},
]).to_csv(SCORES_CSV, index=False)
print(f"Saved -> {SCORES_CSV}")